In [19]:
import numpy as np
import pandas as pd

# ==========================================================
# PARAMETERS
# ==========================================================
m = 1.0
k = 1.5
c = 0.5

# Natural frequency
omega0 = np.sqrt(k / m)

# Driving force parameters
F0 = 1.0                        # force amplitude
freq_cases = [0.5, 1.0, 1.5]    # frequency multipliers

# Time parameters
dt = 0.001
t_max = 50.0
time = np.linspace(0, t_max, int(t_max/dt)+1)
n = len(time)

# Initial conditions
x0 = 1.0
v0 = 0.0

# ==========================================================
# LOOP OVER DRIVING FREQUENCIES
# ==========================================================
for case_num, mult in enumerate(freq_cases, start=1):

    omega_d = mult * omega0

    # ------------------------------------------------------
    # ARRAYS
    # ------------------------------------------------------
    x = np.zeros(n)
    v = np.zeros(n)
    a = np.zeros(n)

    f_spring = np.zeros(n)
    f_damp   = np.zeros(n)
    f_drive  = np.zeros(n)
    f_total  = np.zeros(n)

    ke = np.zeros(n)
    pe = np.zeros(n)
    te = np.zeros(n)

    x[0] = x0
    v[0] = v0

    # ------------------------------------------------------
    # DERIVATIVES
    # ------------------------------------------------------
    def derivatives(t, x_val, v_val):
        fd = F0 * np.sin(omega_d * t)
        dxdt = v_val
        dvdt = (fd - c * v_val - k * x_val) / m
        return dxdt, dvdt

    # ------------------------------------------------------
    # RK4 INTEGRATION
    # ------------------------------------------------------
    for i in range(n - 1):
        t = time[i]

        k1x, k1v = derivatives(t, x[i], v[i])

        k2x, k2v = derivatives(
            t + dt/2,
            x[i] + k1x*dt/2,
            v[i] + k1v*dt/2
        )

        k3x, k3v = derivatives(
            t + dt/2,
            x[i] + k2x*dt/2,
            v[i] + k2v*dt/2
        )

        k4x, k4v = derivatives(
            t + dt,
            x[i] + k3x*dt,
            v[i] + k3v*dt
        )

        x[i+1] = x[i] + (dt/6.0)*(k1x + 2*k2x + 2*k3x + k4x)
        v[i+1] = v[i] + (dt/6.0)*(k1v + 2*k2v + 2*k3v + k4v)

    # ------------------------------------------------------
    # FORCES / ACCELERATION / ENERGY
    # ------------------------------------------------------
    for i in range(n):
        f_drive[i]  = F0 * np.sin(omega_d * time[i])
        f_spring[i] = -k * x[i]
        f_damp[i]   = -c * v[i]
        f_total[i]  = f_spring[i] + f_damp[i] + f_drive[i]
        a[i]        = f_total[i] / m

    ke = 0.5 * m * v**2
    pe = 0.5 * k * x**2
    te = ke + pe

    # ------------------------------------------------------
    # SAVE CSV
    # ------------------------------------------------------
    df = pd.DataFrame({
        "time": time,
        "x": x,
        "v": v,
        "a": a,
        "f_spring": f_spring,
        "f_damp": f_damp,
        "f_drive": f_drive,
        "f_total": f_total,
        "ke": ke,
        "pe": pe,
        "te": te
    })

    filename = f"deterministic_sin_case_{case_num}.csv"
    df.to_csv(filename, index=False)

    print(f"Saved: {filename}")
    print(f"omega0 = {omega0:.4f} | omega_d = {mult:.1f} * omega0 = {omega_d:.4f}")
    # print(f"omega0 = {omega0:.4f}")
    # print(f"omega_drive = {omega_d:.4f}")

Saved: deterministic_sin_case_1.csv
omega0 = 1.2247 | omega_d = 0.5 * omega0 = 0.6124
Saved: deterministic_sin_case_2.csv
omega0 = 1.2247 | omega_d = 1.0 * omega0 = 1.2247
Saved: deterministic_sin_case_3.csv
omega0 = 1.2247 | omega_d = 1.5 * omega0 = 1.8371
